# Phase 2 : Nettoyage des données 

In [19]:
import pandas as pd
donnees = pd.read_csv("../data/raw/fr-en-baccalaureat-par-academie.csv",sep=";", keep_default_na=False)

## 1. Suprimer les colonnes dont on a pas besoin 

In [20]:
donnees = donnees.drop(columns=["Code académie", "Nombre d'admis au 1er groupe","Nombre de refusés au 1er groupe","Nombre d'ajournés, passant les épreuves du 2nd groupe","Nombre d'admis à l'issue du 2nd groupe", "Nombre de refusés à l'issue du 2nd groupe","Nombre d'admis avec mention TB avec les félicitations du jury","Nombre d'admis avec mention TB sans les félicitations du jury","Nombre d'admis avec mention B","Nombre d'admis avec mention AB","Nombre d'admis sans mention"])

## Normalisation des noms de colonnes

In [21]:
#mettre en minuscule
donnees.columns = donnees.columns.str.lower()
donnees.columns

Index(['session', 'libellé académie', 'sexe', 'statut du candidat', 'voie',
       'série', 'diplôme spécialité', 'nombre d'inscrits',
       'nombre de présents', 'nombre d'admis totaux',
       'nombre de refusés totaux'],
      dtype='str')

In [22]:
#Remplacer les espaces par des underscores (_)
donnees.columns = donnees.columns.str.replace(" ", "_")
donnees.columns

Index(['session', 'libellé_académie', 'sexe', 'statut_du_candidat', 'voie',
       'série', 'diplôme_spécialité', 'nombre_d'inscrits',
       'nombre_de_présents', 'nombre_d'admis_totaux',
       'nombre_de_refusés_totaux'],
      dtype='str')

In [23]:
#supprimer les apostrophes
donnees.columns = donnees.columns.str.replace("'", "_")

In [24]:
#enlever les accents 
donnees.columns = donnees.columns.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8")
donnees.columns

Index(['session', 'libelle_academie', 'sexe', 'statut_du_candidat', 'voie',
       'serie', 'diplome_specialite', 'nombre_d_inscrits',
       'nombre_de_presents', 'nombre_d_admis_totaux',
       'nombre_de_refuses_totaux'],
      dtype='str')

## 2. Filtrer les années

In [25]:
# garder que le annes 2024 et 2025
donnees = donnees[donnees["session"].isin([2024, 2025])]
donnees["session"].unique()

array([2024, 2025])

## 3. Filtrer les academies

In [26]:
donnees = donnees[donnees["libelle_academie"].isin(["LILLE", "AMIENS"])]
donnees["libelle_academie"].unique()

<ArrowStringArray>
['AMIENS', 'LILLE']
Length: 2, dtype: str

## 4. Normaliser les noms 

In [27]:
# Passer des MAJ a min
donnees["libelle_academie"] = donnees["libelle_academie"].str.capitalize()
donnees["libelle_academie"].unique()

<ArrowStringArray>
['Amiens', 'Lille']
Length: 2, dtype: str

## 5. Harmonisation 

### 5.1 Colonne sexe 

In [41]:
donnees["sexe"].unique()
donnees["sexe"] = donnees["sexe"].str.replace("FILLES","Féminin")
donnees["sexe"] = donnees["sexe"].str.replace("GARCONS","Masculin")
donnees["sexe"] = donnees["sexe"].str.capitalize()
donnees["sexe"] = donnees["sexe"].str.replace("Feminin","Féminin")
donnees["sexe"].unique()

<ArrowStringArray>
['Féminin', 'Masculin']
Length: 2, dtype: str

### 5.2 Colonne voie

In [29]:
donnees["voie"] = donnees["voie"].str.capitalize()
donnees["voie"].unique()
#donnees["voie"] = donnees["voie"].str.replace("Bac professionnel","Professionnel")
#donnees["voie"] = donnees["voie"].str.replace("Bac technologique","Technologique")
#donnees["voie"] = donnees["voie"].str.replace("Bac general","Général")

<ArrowStringArray>
['Bac professionnel', 'Bac technologique', 'Bac general']
Length: 3, dtype: str

### 5.2 Colonne statut candidat

In [33]:
donnees["statut_du_candidat"] = donnees["statut_du_candidat"].str.capitalize()
donnees["statut_du_candidat"] = donnees["statut_du_candidat"].str.replace(" a "," à ")
donnees["statut_du_candidat"].unique()

<ArrowStringArray>
[          'Apprentissage', 'Enseignement à distance',
      'Formation continue',              'Individuel',
                'Scolaire']
Length: 5, dtype: str

## 6. Vérification

### 6.1 Nombre de lignes et colonnes

In [34]:
donnees.shape

(1286, 11)

Après le nettoyage et le filtrage des données sur les années 2024 et 2025 et les académies de Lille et d’Amiens, le jeu de données est passé de 43 217 lignes et 22 colonnes à 1 286 lignes et 11 colonnes.

### 6.2  Vérification de doublons

In [35]:
donnees.duplicated().sum()

np.int64(0)

Aucun doublon intégral n’a été détecté parmi les 1 286 lignes. Aucune suppression supplémentaire n’est donc nécessaire.

## 6.3 Valeurs manquantes 

In [36]:
donnees.isna().sum()

session                     0
libelle_academie            0
sexe                        0
statut_du_candidat          0
voie                        0
serie                       0
diplome_specialite          0
nombre_d_inscrits           0
nombre_de_presents          0
nombre_d_admis_totaux       0
nombre_de_refuses_totaux    0
dtype: int64

Aucune valeur manquante n’a été détectée dans les 11 colonnes conservées.

## Cohérence des effectifs

### Vérification de la cohérence entre les inscrits et les présents

In [37]:
verif = donnees["nombre_de_presents"] > donnees["nombre_d_inscrits"]
verif.value_counts()

False    1286
Name: count, dtype: int64

### Verif entre les admis et les présents

In [38]:
verif2 = donnees["nombre_d_admis_totaux"] > donnees["nombre_de_presents"]
verif2.value_counts()

False    1286
Name: count, dtype: int64

### Somme des candidats admis et refusés correspond au nombre de candidats présents.

In [40]:
candidats = donnees["nombre_d_admis_totaux"] + donnees["nombre_de_refuses_totaux"]
verif3 = candidats == donnees["nombre_de_presents"]
verif3.value_counts()

True    1286
Name: count, dtype: int64

### Conclusion — Cohérence des effectifs
Les vérifications réalisées sur les 1 286 lignes montrent que :
- Le nombre de présents ne dépasse jamais le nombre d'inscrits.
- Le nombre d'admis ne dépasse jamais le nombre de présents.
- La somme des admis et des refusés correspond toujours au nombre de présents.